# Lab 04 - Bronze Layer: Raw Data Ingestion 
## Building the Foundation of Your Lakehouse


---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Understand** the Bronze Layer role in the Medallion Architecture
2. **Explore** raw JSON data and understand its schema
3. **Implement** batch ingestion with audit columns (`_ingestion_time`, `_source_file`)
4. **Configure** Auto Loader (`cloudFiles`) for incremental file processing
5. **Write** Bronze Delta tables and verify data quality

---

### Medallion Architecture Overview

In [1]:
displayHTML("""
<div style="background: #1B3A4B; border-radius: 12px; padding: 30px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #FFFFFF; letter-spacing: 2px;">MEDALLION ARCHITECTURE</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 8px; flex-wrap: wrap;">
    <div style="text-align: center;">
      <div style="background: #FFFFFF; border: 2px solid #E8E5E0; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #E8E5E0; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #1B3A4B; font-size: 16px;">S</div>
        <div style="font-weight: bold; font-size: 13px; color: #1B1F24;">Source</div>
        <div style="font-size: 11px; color: #757575; margin-top: 4px;">JSON Files<br/>/tmp/</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #CD7F32;">&#8594;</div>
    <div style="text-align: center;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 10px; padding: 16px 20px; min-width: 110px; box-shadow: 0 0 15px rgba(205,127,50,0.3);">
        <div style="width: 36px; height: 36px; background: #CD7F32; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #FFFFFF; font-size: 16px;">B</div>
        <div style="font-weight: bold; font-size: 14px; color: #CD7F32;">BRONZE</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Raw + Audit<br/>Append-Only</div>
        <div style="font-size: 10px; background: #FF3621; color: #FFFFFF; border-radius: 4px; padding: 2px 8px; margin-top: 6px; font-weight: bold; display: inline-block;">YOU ARE HERE</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #757575;">&#8594;</div>
    <div style="text-align: center;">
      <div style="background: #F5F5F5; border: 2px solid #9E9E9E; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #9E9E9E; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #FFFFFF; font-size: 16px;">S</div>
        <div style="font-weight: bold; font-size: 13px; color: #757575;">SILVER</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Cleaned<br/>Validated</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #757575;">&#8594;</div>
    <div style="text-align: center;">
      <div style="background: #FFFDE7; border: 2px solid #F9A825; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #F9A825; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #1B1F24; font-size: 16px;">G</div>
        <div style="font-weight: bold; font-size: 13px; color: #F9A825;">GOLD</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Aggregated<br/>Business-Ready</div>
      </div>
    </div>
  </div>
</div>
""")

MEDALLION ARCHITECTURE 
 
 
 
 S 
 Source 
 JSON Files /tmp/ 
 
 
 → 
 
 
 B 
 BRONZE 
 Raw + Audit Append-Only 
 YOU ARE HERE 
 
 
 → 
 
 
 S 
 SILVER 
 Cleaned Validated 
 
 
 → 
 
 
 G 
 GOLD 
 Aggregated Business-Ready

---
## Step 0: Generate Data

> Run this cell to create placement JSON files and entity reference data.  
> No Unity Catalog or ADLS required — data is written to `/tmp/` and Hive Metastore.
>
> **Important:** This cell intentionally injects data quality issues (nulls, negative values) that you will detect and fix in Lab 05 (Silver Layer).

In [ ]:
# ============================================================
# STEP 0: Generate placement data as JSON files + entities table
# Run this ONCE before starting exercises
# ============================================================

import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col, lit

# --- Cleanup for idempotency ---
for path in [f"/tmp/{USER_DB}/raw", f"/tmp/{USER_DB}/bronze", f"/tmp/{USER_DB}/checkpoints"]:
    try:
        dbutils.fs.rm(path, recurse=True)
    except:
        pass

# Derive a unique database name per participant
_user = spark.sql("SELECT current_user()").first()[0]
_user_clean = _user.split("@")[0].replace(".", "_").replace("-", "_").lower()
USER_DB = f"training_lab_{_user_clean}"
print(f"Your personal database: {USER_DB}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {USER_DB}")

# --- AXA Entities (18 records) ---
entities_data = [
    ("AXA-FR-001", "AXA France", "Insurance", "Europe", "France"),
    ("AXA-FR-002", "AXA France IARD", "Insurance", "Europe", "France"),
    ("AXA-DE-001", "AXA Germany", "Insurance", "Europe", "Germany"),
    ("AXA-UK-001", "AXA UK", "Insurance", "Europe", "United Kingdom"),
    ("AXA-BE-001", "AXA Belgium", "Insurance", "Europe", "Belgium"),
    ("AXA-CH-001", "AXA Switzerland", "Insurance", "Europe", "Switzerland"),
    ("AXA-IT-001", "AXA Italy", "Insurance", "Europe", "Italy"),
    ("AXA-ES-001", "AXA Spain", "Insurance", "Europe", "Spain"),
    ("AXA-US-001", "AXA XL US", "Reinsurance", "Americas", "United States"),
    ("AXA-US-002", "AXA Equitable", "Life Insurance", "Americas", "United States"),
    ("AXA-MX-001", "AXA Mexico", "Insurance", "Americas", "Mexico"),
    ("AXA-BR-001", "AXA Brazil", "Insurance", "Americas", "Brazil"),
    ("AXA-JP-001", "AXA Japan", "Insurance", "Asia Pacific", "Japan"),
    ("AXA-HK-001", "AXA Hong Kong", "Insurance", "Asia Pacific", "Hong Kong"),
    ("AXA-SG-001", "AXA Singapore", "Insurance", "Asia Pacific", "Singapore"),
    ("AXA-XL-001", "AXA XL Reinsurance", "Reinsurance", "Global", "Bermuda"),
    ("AXA-IM-001", "AXA Investment Managers", "Asset Management", "Global", "France"),
    ("AXA-GO-001", "AXA Global Operations", "Operations", "Global", "France")
]
entities_schema = StructType([
    StructField("entity_id", StringType()), StructField("entity_name", StringType()),
    StructField("entity_type", StringType()), StructField("region", StringType()),
    StructField("country", StringType())
])
spark.createDataFrame(entities_data, entities_schema) \
    .write.mode("overwrite").saveAsTable(f"{USER_DB}.axa_entities")

# --- Placement data generation ---
entity_ids = [e[0] for e in entities_data]
asset_classes = ["EQUITY", "BOND", "REAL_ESTATE", "ALTERNATIVE", "CASH", "DERIVATIVES"]
currencies = ["EUR", "USD", "GBP", "CHF"]

placements_schema = StructType([
    StructField("placement_id", StringType()),
    StructField("axa_entity_id", StringType()),
    StructField("asset_class", StringType()),
    StructField("instrument_id", StringType()),
    StructField("market_value", DoubleType()),
    StructField("book_value", DoubleType()),
    StructField("currency", StringType()),
    StructField("valuation_date", StringType())
])

def generate_placements(start_id, count, inject_quality_issues=False):
    data = []
    for i in range(start_id, start_id + count):
        pid = f"PLC-{str(i).zfill(6)}"
        eid = random.choice(entity_ids)
        ac = random.choice(asset_classes)
        iid = f"ISIN-{str(random.randint(1, 200)).zfill(4)}"
        mv = round(random.uniform(500000, 15000000), 2)
        bv = round(random.uniform(500000, 15000000), 2)
        cur = random.choice(currencies)
        day_offset = random.randint(0, 364)
        vdate = (datetime(2024, 1, 1) + timedelta(days=day_offset)).strftime("%Y-%m-%d")

        if inject_quality_issues:
            r = random.random()
            if r < 0.05:
                mv = None
            elif r < 0.08:
                mv = round(-random.uniform(100000, 2000000), 2)

            r2 = random.random()
            if r2 < 0.02:
                bv = None
            elif r2 < 0.03:
                bv = round(-random.uniform(1000, 50000), 2)

            if random.random() < 0.01:
                vdate = None

        data.append((pid, eid, ac, iid, mv, bv, cur, vdate))
    return data

# Initial batch: 300 records WITH quality issues
random.seed(42)
initial_data = generate_placements(1, 300, inject_quality_issues=True)
spark.createDataFrame(initial_data, placements_schema) \
    .write.mode("overwrite").json(f"/tmp/{USER_DB}/raw/placements")

# Incremental batch 1: 50 records (clean)
inc1_data = generate_placements(301, 50, inject_quality_issues=False)
spark.createDataFrame(inc1_data, placements_schema) \
    .coalesce(1).write.mode("overwrite").json(f"/tmp/{USER_DB}/raw/placements_incremental/batch_1")

# Incremental batch 2: 30 records (clean)
inc2_data = generate_placements(351, 30, inject_quality_issues=False)
spark.createDataFrame(inc2_data, placements_schema) \
    .coalesce(1).write.mode("overwrite").json(f"/tmp/{USER_DB}/raw/placements_incremental/batch_2")

print("=" * 60)
print("DATA GENERATED")
print("=" * 60)
print(f"Entities:       {spark.table(f'{USER_DB}.axa_entities').count()} rows (Hive table)")
print(f"Placements:     300 records -> /tmp/training_lab/raw/placements/")
print(f"Incremental:    80 records  -> /tmp/training_lab/raw/placements_incremental/")
print(f"")
print(f"NOTE: Initial batch includes intentional quality issues")
print(f"      (null values, negative amounts) for Silver Layer exercises.")

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

# ---- Configuration (Hive Metastore, /tmp/ paths) ----
DATABASE_NAME = USER_DB
SOURCE_PATH = f"/tmp/{USER_DB}/raw/placements"
SOURCE_INCREMENTAL = f"/tmp/{USER_DB}/raw/placements_incremental"
BRONZE_PATH = f"/tmp/{USER_DB}/bronze/placements"
CHECKPOINT_PATH = f"/tmp/{USER_DB}/checkpoints/bronze_placements"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")

# Verify source files
files = dbutils.fs.ls(SOURCE_PATH)
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Database:       {DATABASE_NAME}")
print(f"Source Path:    {SOURCE_PATH}")
print(f"Incremental:    {SOURCE_INCREMENTAL}")
print(f"Bronze Path:    {BRONZE_PATH}")
print(f"Checkpoint:     {CHECKPOINT_PATH}")
print(f"Source files:   {len(files)} items")
for f in files[:5]:
    print(f"  - {f.name} ({f.size} bytes)")

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Une cellule de configuration bien structuree avec des noms de variables clairs rend votre notebook portable et facile a debugger.</span>
</div>
""")

A retenir : Une cellule de configuration bien structuree avec des noms de variables clairs rend votre notebook portable et facile a debugger.

---

## Section 2: Understanding the Bronze Layer

> *"La couche Bronze est votre police d'assurance. En conservant les donnees exactement telles qu'elles arrivent de la source, vous pouvez toujours remonter a l'original si quelque chose tourne mal en aval."*

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Data Flow: Source to Bronze</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 0; flex-wrap: wrap;">
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #1B1F24;">Source</div>
        <div style="font-size: 11px; color: #757575;">JSON dans /tmp/</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #CD7F32; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 8px; padding: 14px 20px; min-width: 110px; box-shadow: 0 4px 12px rgba(205,127,50,0.3);">
        <div style="font-weight: bold; font-size: 15px; color: #CD7F32;">BRONZE</div>
        <div style="font-size: 11px; color: #1B1F24;">Raw + Metadata</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #757575; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px dashed #757575; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #757575;">Silver</div>
        <div style="font-size: 11px; color: #757575;">Lab 05</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #757575; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px dashed #757575; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #757575;">Gold</div>
        <div style="font-size: 11px; color: #757575;">Lab 06</div>
      </div>
    </div>
  </div>
</div>
""")

Data Flow: Source to Bronze 
 
 
 
 Source 
 JSON dans /tmp/ 
 
 
 ➔ 
 
 
 BRONZE 
 Raw + Metadata 
 
 
 ➔ 
 
 
 Silver 
 Lab 05 
 
 
 ➔ 
 
 
 Gold 
 Lab 06

### Bronze Layer Principles

| Principle | Description |
|-----------|-------------|
| **Raw Data** | Store data exactly as received from the source, no business transformations |
| **Audit Columns** | Add metadata like `_ingestion_time`, `_source_file` for traceability |
| **Append-Only** | Never update or delete; each ingestion adds new records |
| **No Transformations** | Cleaning, validation, and enrichment happen in Silver |
| **Schema Preservation** | Keep the original schema; handle evolution with `mergeSchema` |

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">La couche Bronze sert de <em>source unique de verite</em> pour les donnees brutes. En preservant le format original et en ajoutant uniquement des metadonnees d'audit, vous maintenez une tracabilite complete.</span>
</div>
""")

A retenir : La couche Bronze sert de source unique de verite pour les donnees brutes. En preservant le format original et en ajoutant uniquement des metadonnees d'audit, vous maintenez une tracabilite complete.

---

## Section 3: Batch Ingestion

Dans cette section, nous allons lire les fichiers JSON bruts, ajouter des colonnes d'audit pour la tracabilite, et ecrire le resultat en tant que table Delta dans la couche Bronze.

### Exercise 3.1: Explore Raw Data

Avant d'ingerer, explorons la structure des donnees source.

In [ ]:
# Read the raw JSON data to explore schema and content
sample_df = spark.read \
    .format("json") \
    .load(SOURCE_PATH)

print("Schema of raw placements data:")
sample_df.printSchema()

print(f"Total records: {sample_df.count()}")
display(sample_df.limit(5))

### Exercise 3.2: Add Audit Columns

Les colonnes d'audit sont essentielles pour la tracabilite. Elles indiquent **quand** les donnees ont ete ingerees et **d'ou** elles proviennent.

| Column | Function | Purpose |
|--------|----------|---------|
| `_ingestion_time` | `current_timestamp()` | When the record was ingested |
| `_source_file` | `input_file_name()` | Which file the record came from |
| `_processing_time` | `current_timestamp()` | When the record was processed |

**TODO:** Fill in the blanks with the correct PySpark functions.

In [ ]:
# TODO: Fill in the blanks to add audit columns

raw_df = spark.read \
    .format("json") \
    .load(SOURCE_PATH)

bronze_df = raw_df \
    .withColumn("_ingestion_time", ________) \
    .withColumn("_source_file", ________) \
    .withColumn("_processing_time", current_timestamp())

print("Schema with audit columns:")
bronze_df.printSchema()
display(bronze_df.select("placement_id", "_ingestion_time", "_source_file").limit(3))

In [ ]:
# Verification 3.2
assert "_ingestion_time" in bronze_df.columns, "_ingestion_time column missing - use current_timestamp()"
assert "_source_file" in bronze_df.columns, "_source_file column missing - use input_file_name()"
_v_src = bronze_df.select("_source_file").first()[0]
assert _v_src is not None and len(str(_v_src)) > 0, "_source_file is empty - ensure input_file_name() is used"
print(f"Exercise 3.2 passed! Audit columns added: _ingestion_time, _source_file, _processing_time")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">Le premier blanc a besoin d'une fonction qui retourne la date et l'heure actuelles : <code>current_timestamp()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Le deuxieme blanc a besoin d'une fonction qui retourne le chemin du fichier source : <code>input_file_name()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Les deux fonctions ont ete importees dans la cellule de configuration</li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 Le premier blanc a besoin d'une fonction qui retourne la date et l'heure actuelles : current_timestamp() 
 Le deuxieme blanc a besoin d'une fonction qui retourne le chemin du fichier source : input_file_name() 
 Les deux fonctions ont ete importees dans la cellule de configuration

### Exercise 3.3: Write to Bronze Delta Table

Ecrivons les donnees enrichies au format Delta et enregistrons-les en tant que table Hive.

| Parameter | Value | Why |
|-----------|-------| --- |
| `format` | `"delta"` | Delta Lake provides ACID transactions, versioning, and time travel |
| `mode` | `"overwrite"` | First load replaces any existing data |

**TODO:** Fill in the blanks for the write format and mode.

In [ ]:
# TODO: Fill in the blanks for format and mode

bronze_df.write \
    .format(________) \
    .mode(________) \
    .option("overwriteSchema", "true") \
    .save(BRONZE_PATH)

# Register as Hive table
spark.sql(f"DROP TABLE IF EXISTS {DATABASE_NAME}.bronze_placements")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.bronze_placements
    USING DELTA LOCATION '{BRONZE_PATH}'
""")

result_df = spark.table(f"{DATABASE_NAME}.bronze_placements")
print(f"Bronze table records: {result_df.count()}")
display(result_df.limit(5))

In [ ]:
# Verification 3.3
_v_count = spark.table(f"{DATABASE_NAME}.bronze_placements").count()
assert _v_count > 0, "Bronze table is empty - check format ('delta') and mode ('overwrite')"
assert _v_count >= 290, f"Expected ~300 records, got {_v_count}"
print(f"Exercise 3.3 passed! Bronze Delta table has {_v_count} records")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">Le format est celui que vous avez utilise dans le Lab 03 : <code>"delta"</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Le mode pour un chargement initial doit remplacer les donnees existantes : <code>"overwrite"</code></li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 Le format est celui que vous avez utilise dans le Lab 03 : "delta" 
 Le mode pour un chargement initial doit remplacer les donnees existantes : "overwrite"

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">L'ingestion batch est ideale pour le chargement initial. Utilisez <code>mode("overwrite")</code> pour le premier chargement et <code>mode("append")</code> pour les chargements incrementaux.</span>
</div>
""")

A retenir : L'ingestion batch est ideale pour le chargement initial. Utilisez mode("overwrite") pour le premier chargement et mode("append") pour les chargements incrementaux.

---

## Section 4: Auto Loader for Incremental Ingestion

> *"En production, de nouveaux fichiers arrivent en continu. Auto Loader detecte automatiquement les nouveaux fichiers, les traite exactement une fois, et gere les changements de schema — le tout sans ecrire de logique complexe de suivi de fichiers."*

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Auto Loader: How It Works</h3>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 30px; flex-wrap: wrap; margin-top: 15px;">
    <div style="text-align: center;">
      <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 15px;">
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 12px 18px;">
          <div style="font-weight: bold; color: #1B1F24;">New Files</div>
          <div style="font-size: 11px; color: #757575;">/tmp/ paths</div>
        </div>
        <div style="font-size: 24px; color: #FF3621;">&#10132;</div>
        <div style="background: #1B3A4B; border-radius: 8px; padding: 12px 18px; color: white;">
          <div style="font-weight: bold; color: #FFFFFF;">Auto Loader</div>
          <div style="font-size: 11px; color: #F9F7F4;">cloudFiles</div>
        </div>
        <div style="font-size: 24px; color: #FF3621;">&#10132;</div>
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 12px 18px;">
          <div style="font-weight: bold; color: #1B1F24;">Delta Table</div>
          <div style="font-size: 11px; color: #757575;">Bronze Layer</div>
        </div>
      </div>
      <div style="display: flex; justify-content: center;">
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 10px 18px;">
          <div style="font-weight: bold; color: #1B3A4B;">Checkpoint</div>
          <div style="font-size: 11px; color: #757575;">Tracks processed files</div>
        </div>
      </div>
    </div>
    <div style="background: #FFFFFF; border-radius: 8px; padding: 15px 20px; min-width: 220px; border: 1px solid #E0E0E0;">
      <div style="font-weight: bold; color: #1B3A4B; margin-bottom: 8px;">Key Features</div>
      <div style="font-size: 13px; color: #1B1F24; line-height: 1.8;">
        &#10003; Automatic file discovery<br/>
        &#10003; Schema inference and evolution<br/>
        &#10003; Exactly-once processing<br/>
        &#10003; Handles millions of files<br/>
        &#10003; Fault-tolerant with checkpoints
      </div>
    </div>
  </div>
</div>
""")

Auto Loader: How It Works 
 
 
 
 
 New Files 
 /tmp/ paths 
 
 ➔ 
 
 Auto Loader 
 cloudFiles 
 
 ➔ 
 
 Delta Table 
 Bronze Layer 
 
 
 
 
 Checkpoint 
 Tracks processed files 
 
 
 
 
 Key Features 
 
 ✓ Automatic file discovery 
 ✓ Schema inference and evolution 
 ✓ Exactly-once processing 
 ✓ Handles millions of files 
 ✓ Fault-tolerant with checkpoints

### Exercise 4.1: Configure Auto Loader

Auto Loader utilise le format `cloudFiles` pour configurer une source streaming qui detecte automatiquement les nouveaux fichiers.

| Option | Value | Purpose |
|--------|-------|---------|
| `format` | `"cloudFiles"` | Tells Spark to use Auto Loader |
| `cloudFiles.format` | `"json"` | Format of the source files |
| `cloudFiles.schemaLocation` | checkpoint path | Where to store inferred schema |
| `cloudFiles.inferColumnTypes` | `"true"` | Infer proper data types |
| `cloudFiles.useIncrementalListing` | `"auto"` | File discovery mode |

**TODO:** Fill in the blank with the correct format name.

In [ ]:
# TODO: Fill in the blank with the Auto Loader format

auto_loader_df = spark.readStream \
    .format(________) \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.useIncrementalListing", "auto") \
    .load(SOURCE_INCREMENTAL)

print("Auto Loader stream configured successfully!")

In [ ]:
# Verification 4.1
assert auto_loader_df.isStreaming, "DataFrame is not streaming - check the format (should be 'cloudFiles')"
print(f"Exercise 4.1 passed! Auto Loader stream configured with cloudFiles format")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">Le format d'Auto Loader est un nom compose de deux mots : <code>"cloudFiles"</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">C'est un format specifique a Databricks pour l'ingestion incrementale</li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 Le format d'Auto Loader est un nom compose de deux mots : "cloudFiles" 
 C'est un format specifique a Databricks pour l'ingestion incrementale

### Exercise 4.2: Add Audit Columns to Stream

Comme pour l'ingestion batch, les donnees en streaming ont aussi besoin de colonnes d'audit.

> **Note:** Pour le streaming (Auto Loader), utilisez `col("_metadata.file_path")` au lieu de `input_file_name()`.

In [ ]:
# Add audit columns to the streaming DataFrame
# Note: For streaming, use col("_metadata.file_path") instead of input_file_name()

bronze_stream_df = auto_loader_df \
    .withColumn("_ingestion_time", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("_processing_time", current_timestamp())

print("Audit columns added to stream")

### Exercise 4.3: Write Stream to Delta

Ecrivons le flux en mode `append` vers la table Bronze Delta. Le trigger `availableNow=True` traite tous les fichiers disponibles puis s'arrete.

| Parameter | Value | Why |
|-----------|-------| --- |
| `format` | `"delta"` | Write to Delta Lake |
| `outputMode` | `"append"` | Add new records without overwriting |
| `checkpointLocation` | checkpoint path | Track which files have been processed |
| `mergeSchema` | `"true"` | Accept new columns if schema evolves |
| `trigger` | `availableNow=True` | Process all available, then stop |

**TODO:** Fill in the blank for the output format.

In [ ]:
# TODO: Fill in the blank with the correct write format

query = bronze_stream_df.writeStream \
    .format(________) \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(BRONZE_PATH)

query.awaitTermination()
print("Auto Loader ingestion completed!")

In [ ]:
# Verification 4.3
_v_total = spark.read.format("delta").load(BRONZE_PATH).count()
assert _v_total > 300, f"Expected > 300 records after incremental ingestion, got {_v_total}"
print(f"Exercise 4.3 passed! Bronze table has {_v_total} records (initial batch + incremental)")

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Auto Loader (<code>cloudFiles</code>) est la methode recommandee pour ingerer des fichiers de maniere incrementale dans Databricks. Il suit automatiquement les fichiers traites via les checkpoints, garantit un traitement exactly-once, et gere l'evolution du schema.</span>
</div>
""")

A retenir : Auto Loader ( cloudFiles ) est la methode recommandee pour ingerer des fichiers de maniere incrementale dans Databricks. Il suit automatiquement les fichiers traites via les checkpoints, garantit un traitement exactly-once, et gere l'evolution du schema.

---

## Section 5: Verify Bronze Layer

Inspectons la table Bronze pour confirmer que tout est correct avant de passer a la couche Silver.

In [ ]:
# View Delta table history
print("Delta Table History:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`"))

In [ ]:
# Check data distribution by asset class
print("Records by Asset Class:")
display(
    spark.read.format("delta").load(BRONZE_PATH)
    .groupBy("asset_class")
    .count()
    .orderBy("count", ascending=False)
)

In [ ]:
# Check data distribution by entity
print("Records by Entity:")
display(
    spark.read.format("delta").load(BRONZE_PATH)
    .groupBy("axa_entity_id")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

In [ ]:
# Preview data quality issues (will be handled in Silver layer)
from pyspark.sql.functions import count, when, sum as spark_sum

bronze_final = spark.read.format("delta").load(BRONZE_PATH)
print("Data Quality Preview (these issues will be fixed in Silver layer):")
display(
    bronze_final.select(
        count("*").alias("total_records"),
        spark_sum(when(col("market_value").isNull(), 1).otherwise(0)).alias("null_market_value"),
        spark_sum(when(col("book_value").isNull(), 1).otherwise(0)).alias("null_book_value"),
        spark_sum(when(col("valuation_date").isNull(), 1).otherwise(0)).alias("null_valuation_date"),
        spark_sum(when(col("market_value") < 0, 1).otherwise(0)).alias("negative_market_value")
    )
)

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Verifiez toujours votre couche Bronze apres l'ingestion. Les problemes de qualite (nulls, valeurs negatives) sont normaux a ce stade — le nettoyage se fait dans Silver.</span>
</div>
""")

A retenir : Verifiez toujours votre couche Bronze apres l'ingestion. Les problemes de qualite (nulls, valeurs negatives) sont normaux a ce stade — le nettoyage se fait dans Silver.

---

## What You Learned

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Lab 04 Summary</h3>
  <table style="width: 100%; border-collapse: collapse; background: #FFFFFF; border-radius: 8px; overflow: hidden;">
    <thead>
      <tr style="background: #1B3A4B; color: #FFFFFF;">
        <th style="padding: 12px 15px; text-align: left;">Concept</th>
        <th style="padding: 12px 15px; text-align: left;">What You Learned</th>
        <th style="padding: 12px 15px; text-align: left;">Key Command / API</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Bronze Layer</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Raw data ingestion with audit metadata</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>.write.format("delta")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Audit Columns</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Track data lineage and source provenance</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>current_timestamp()</code>, <code>input_file_name()</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Auto Loader</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Incremental file processing with automatic detection</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>.format("cloudFiles")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Checkpointing</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Exactly-once processing and fault tolerance</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>checkpointLocation</code> option</td>
      </tr>
      <tr style="background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Data Quality</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Bronze preserves quality issues — cleaning happens in Silver</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Profiling with <code>isNull()</code>, range checks</td>
      </tr>
    </tbody>
  </table>
</div>
""")

Concept,What You Learned,Key Command / API
Bronze Layer,Raw data ingestion with audit metadata,".write.format(""delta"")"
Audit Columns,Track data lineage and source provenance,"current_timestamp(), input_file_name()"
Auto Loader,Incremental file processing with automatic detection,".format(""cloudFiles"")"
Checkpointing,Exactly-once processing and fault tolerance,checkpointLocation option
Data Quality,Bronze preserves quality issues — cleaning happens in Silver,"Profiling with isNull(), range checks"


### Next Steps

Passez au **Lab 05 - Silver Layer** ou vous allez :
- Lire les donnees Bronze que vous venez de creer
- **Profiler** les problemes de qualite (nulls, valeurs negatives)
- **Nettoyer** les donnees (filtres, validation de plage)
- **Enrichir** en joignant avec la table de reference des entites
- **Calculer** des metriques business (gain/perte non realise)
- Ecrire des tables Silver Delta pretes pour la production

---
## Cleanup

In [ ]:
# Cleanup (optional -- uncomment to reset)
# spark.sql(f"DROP TABLE IF EXISTS {DATABASE_NAME}.bronze_placements")
# spark.sql(f"DROP TABLE IF EXISTS {DATABASE_NAME}.axa_entities")
# dbutils.fs.rm("/tmp/{USER_DB}", recurse=True)
# print("Lab 04 cleanup complete.")